# 01 — Préparation et indexation du corpus

**Objectif** : transformer les 351 offres d'emploi IT (étape 1) en un index vectoriel interrogeable.

Ce notebook couvre les points **2.1** et **2.2** du formulaire SpeciTec :
- 2.1 Nature et volume des sources
- 2.2 Préparation des données (chunking, embedding, vector store)

---

## Décisions de design

| Décision | Choix retenu | Justification |
|----------|-------------|---------------|
| Granularité | 1 offre = 1 document | Offres courtes (~200 mots), pas besoin de chunking |
| Embedding model | `paraphrase-multilingual-MiniLM-L12-v2` | Supporte FR + EN, 384 dims, CPU rapide |
| Vector store | FAISS `IndexFlatIP` | Exact search, persistant, compatible Python 3.14 |
| Filtrage | Exclut `NOT_RELEVANT` | Garde DATA_ENGINEERING, BI_ANALYTICS, DBA_INFRA |

In [ ]:
import sys
from pathlib import Path

# Ajoute la racine du projet au path pour les imports
ROOT = Path().resolve().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print('Racine projet :', ROOT)

## 2.1 — Nature et volume des sources

In [ ]:
import pandas as pd
from config import CORPUS_PATH, RELEVANT_CATEGORIES

# Chargement du corpus complet (avant filtrage)
df_all = pd.read_csv(CORPUS_PATH, encoding='utf-8')
df_all['description'] = df_all['description'].fillna('')

print('=== 2.1 Nature et volume des sources ===')
print(f'Total offres collectées   : {len(df_all)}')
print(f'Source                    : Adzuna API (offres d\'emploi IT Suisse romande)')
print(f'Date de collecte          : {df_all["date_collected"].max()}')
print(f'Langues                   : FR + EN (corpus mixte)')
print()
print('Distribution par catégorie (étiquetage étape 1) :')
print(df_all['label'].value_counts())
print()

# Statistiques sur la longueur des descriptions
df_all['desc_words'] = df_all['description'].str.split().str.len()
print('Longueur des descriptions (mots) :')
print(df_all['desc_words'].describe().round(1))

## 2.2 — Préparation des données

### Filtrage et formatage

In [ ]:
from src.indexer import load_corpus, format_document

# Chargement avec filtrage NOT_RELEVANT
df = load_corpus()

print(f'Offres indexées (pertinentes) : {len(df)}')
print(f'Offres exclues (NOT_RELEVANT) : {len(df_all) - len(df)}')
print()
print('Distribution des offres indexées :')
print(df['label'].value_counts())
print()

# Aperçu du format de document
print('=== Exemple de document formaté ===')
print(format_document(df.iloc[0].to_dict()))

### Génération des embeddings et indexation FAISS

In [ ]:
from src.indexer import build_index

# Lance l'indexation complète
# Premier lancement : télécharge le modèle (~420MB, une seule fois)
stats = build_index(verbose=True)

### Vérification de l'index persisté

In [ ]:
import json
import faiss
import numpy as np
from config import FAISS_INDEX_PATH, FAISS_META_PATH

# Rechargement depuis disque — vérifie la persistance
index = faiss.read_index(str(FAISS_INDEX_PATH))
metadata = json.loads(FAISS_META_PATH.read_text(encoding='utf-8'))

print(f'Index rechargé     : {index.ntotal} vecteurs')
print(f'Dimension          : {index.d}')
print(f'Métadonnées        : {len(metadata)} entrées')
print(f'Cohérence          : {index.ntotal == len(metadata)}')
print()
print('Exemple de métadonnée (offre #0) :')
print(json.dumps(metadata[0], ensure_ascii=False, indent=2))

### Test de recherche basique (smoke test)

In [ ]:
from sentence_transformers import SentenceTransformer
from config import EMBEDDING_MODEL

model = SentenceTransformer(EMBEDDING_MODEL)

query = "data engineer dbt Genève"
q_vec = model.encode([query], normalize_embeddings=True).astype(np.float32)

scores, indices = index.search(q_vec, 3)

print(f'Requête : "{query}"')
print()
for rank, (idx, score) in enumerate(zip(indices[0], scores[0]), 1):
    m = metadata[idx]
    print(f'[{rank}] Score={score:.4f} | {m["title"]} — {m["company"]} | {m["location"]}')
    print(f'     Catégorie : {m["label"]}')

---

## Récapitulatif Phase 1

| Paramètre | Valeur |
|-----------|--------|
| Documents indexés | voir `stats` |
| Modèle embedding | `paraphrase-multilingual-MiniLM-L12-v2` |
| Dimension vecteurs | 384 |
| Similarité | Cosine (produit scalaire après normalisation L2) |
| Index type | `faiss.IndexFlatIP` (exact search) |
| Persistance | `data/vectorstore/jobs.index` + `jobs_meta.json` |

**Prochaine étape** : `02_retrieval.ipynb` — recherche sémantique + re-ranking